# Full pipeline: from raw data to results

Use this only to rebuild everything from scratch (inventory + Earth Engine export). To re-run the experiments from the archived data, use **reproduce_results.ipynb** instead.

Requirements: a Google account registered for non-commercial Earth Engine use, a Cloud project, and the **Earth Engine API** and **Google Drive API** enabled in that project.

## 1. Upload the code and set your Cloud project

In [ ]:
GEE_PROJECT = "bikalpa-landslide"   # your Earth Engine Cloud project ID

from google.colab import files
import zipfile, os
up = files.upload()                      # choose landslide_farwest_nepal.zip
zipfile.ZipFile([f for f in up if f.endswith('.zip')][0]).extractall('/content')
os.chdir('/content/landslide_farwest_nepal')
!pip -q install -r requirements.txt
!sed -i 's/GEE_PROJECT = ".*"/GEE_PROJECT = "{GEE_PROJECT}"/' config.py
!grep GEE_PROJECT config.py

## 2. Download the landslide inventory (Zenodo)

In [ ]:
!python 02_prepare_data.py --download-only

## 3. Sign in to Earth Engine
Click the link, choose your project, keep **read-only scopes unticked**, generate the token and paste it below.

In [ ]:
import ee
ee.Authenticate(auth_mode='notebook', force=True)
ee.Initialize(project=GEE_PROJECT)
print(ee.String('Earth Engine is working').getInfo())

## 4. Start the Earth Engine export (runs on Google's servers, 20–60 minutes)

In [ ]:
!python 01_gee_export.py

## 5. Check the export status
Re-run this cell until it shows **COMPLETED**.

In [ ]:
ee.Initialize(project=GEE_PROJECT)
for t in ee.batch.Task.list()[:3]:
    s = t.status(); print(s['description'], '->', s['state'], s.get('error_message', ''))

## 6. Download the export from Drive into the project

In [ ]:
import io
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
svc = build('drive', 'v3', credentials=ee.data.get_persistent_credentials())
found = svc.files().list(q="name contains 'fw_stack' and trashed=false", fields='files(id,name,size)').execute()['files']
print('Found:', found)
os.makedirs('data/gee', exist_ok=True)
for f in found:
    if f['name'].endswith('.tif'):
        with io.FileIO(f"data/gee/{f['name']}", 'wb') as fh:
            dl = MediaIoBaseDownload(fh, svc.files().get_media(fileId=f['id']), chunksize=50*1024*1024)
            done = False
            while not done:
                st, done = dl.next_chunk(); print(f['name'], int(st.progress()*100), '%')
!ls -lh data/gee

## 7. Build features and samples
`--skip-osm` matches the published study (no distance-to-road factor).

In [ ]:
!python 02_prepare_data.py --skip-download --skip-osm

## 8. Back up the processed data to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/landslide_farwest/backup
!cp data/processed/samples.csv data/processed/dated_samples.csv data/processed/feature_names.json data/processed/features.tif /content/drive/MyDrive/landslide_farwest/backup/

## 9. Experiments, map, check and backup (about 90 minutes)

In [ ]:
!python 03_experiments.py
import re
best = re.search(r'Best model under spatial CV: \*\*(\w+)\*\*', open('results/results_for_paper.md').read()).group(1)
!sed -i 's/MAP_MODEL = ".*"/MAP_MODEL = "{best}"/' config.py
!python 04_map.py
!python verify_results.py
!zip -qr results.zip results
!cp -r results results.zip /content/drive/MyDrive/landslide_farwest/backup/